In [4]:
import numpy as np
import sys
sys.path.append("/home/steven/thesis/mod")
from model_independent import model_independent
from model_independent import Growth_Factor

In [ ]:
class postprocess:
    def __init__(self, chain_files):
        # Setting up indices to read the chain file
        self.chain_files = chain_files
        self.mc = {}
        self.sample_names = {}
        self.sample_values = {}
        self.samples = {}
        for key in chain_files :
            index_h0 = None # Some chains have fixed h0
            file = chain_files[key] 
            with open(file, "r") as f:
                header = f.readline().strip().split()
                for i,name in enumerate(header) :
                    if "omch2" in name :
                        index_omch2 = i
                        
                    if "ombh2" in name :
                        index_ombh2 = i
                    if "s_8" in name :
                        index_s8 = i
                    if "h0" in name :
                        index_h0 = i
                    if "intrinsic_alignment" in name :
                        index_ia = i
                        
            # Reading the chain file
            data = np.loadtxt(file)
            self.prior = data[:,-4]
            self.like = data[:,-3]
            self.post = data[:,-2]
            self.w = data[:,-1]
            self.omch2 = data[:,index_omch2]
            self.ombh2 = data[:,index_ombh2]
            self.s8 = data[:,index_s8]
            self.ia = data[:,index_ia]
            if index_h0 is None :
                self.h0 = 0.735
            else :
                self.h0 = data[:,index_h0]
            self.om = (self.omch2+self.ombh2)/self.h0**2
            self.sigma_8 = self.s8/(self.om/0.3)**0.5

            ### Fitting for alpha by performing linear regression : log(sigma_8) = -alpha*log(omega_m/0.3) + c
            X = np.log(self.om/0.3)
            Y = np.log(self.sigma_8)
            coeffs, cov = np.polyfit(X,Y,deg=1,cov=True,w=self.w)
            self.alpha,Sigma_8_reg = coeffs
            self.alpha = -self.alpha
            self.alpha_err , Sigma_8_err = np.sqrt(np.diag(cov))
            # print(f"alpha = {self.alpha} +/- {self.alpha_err}")
            # print(f"Sigma_8 (from regression) = {np.exp(Sigma_8_reg)} +/- {Sigma_8_err}")
            self.Sigma_8 = self.sigma_8*(self.om/0.3)**self.alpha

            if index_h0 is None :
                self.sample_values[key] = np.vstack([self.om,self.sigma_8,self.s8,self.Sigma_8,self.ia,self.omch2,self.ombh2]).T
                self.sample_names[key] = ["\\Omega_m","\\sigma_8","S_8","\\Sigma_8","IA","\Omega_ch^2","\Omega_bh^2"]
            else : 
                self.sample_values[key] = np.vstack([self.om,self.sigma_8,self.s8,self.Sigma_8,self.ia,self.omch2,self.ombh2,self.h0]).T
                self.sample_names[key] = ["\\Omega_m","\\sigma_8","S_8","\\Sigma_8","IA","\Omega_ch^2","\Omega_bh^2","h"]
            self.samples[key] = np.vstack([self.sample_names,self.sample_values])
            self.mc[key] = MCSamples(samples=self.sample_values[key], weights=self.w, names=self.sample_names[key], labels=self.sample_names[key])   

    ### Function for getting modes and uncertainties
    def best_fit(self,samples=None,weights=None) : 
        if samples is None :
            samples = self.samples
            weights = self.w
            
        print("Best-fit values (modes and 68% HPDI uncertainties) :")
        for i in range(samples[0].shape[1]):
            name = samples[0,i]
            values = samples[1:,i].astype(float)
            ## Getting the mode
            hist, bin_edges = np.histogram(values,bins=200,weights=weights,density=True)
            bin_centers = 0.5*(bin_edges[1:]+bin_edges[:-1])
            mode = bin_centers[np.argmax(hist)]

            ## Getting the uncertainty using HPDI
            sort = np.argsort(values)
            sorted_values = values[sort]
            sorted_w = weights[sort]
            cdf = np.cumsum(sorted_w)
            intervals = []
            for i in range(len(weights)) :
                target_cdf = cdf[i]+0.68
                if target_cdf>1 :
                    break
                target_index = np.argmin(np.abs(cdf-target_cdf))
                interval = sorted_values[target_index] - sorted_values[i]
                intervals.append(interval)
            interval = np.min(intervals)
            uncertainty = interval
            print(f"{name} = {mode} +/- {uncertainty}")

    ### Function for plotting contour
    def plot_contour(self, wanted_parameters, wanted_chains, samples=None, weights=None) :
        if samples is None :
            mc = self.mc
            names = self.sample_names
        else :
            names = samples[0]
            samples = samples[1:].astype(float)
            mc = MCSamples(samples=samples, weights=weights, names=names, labels=names)
        list_mc = []
        for key in self.chain_files :
            dummy = key
            if key in wanted_chains :
                list_mc.append(mc[key])
        for i in range(len(names)-1) :
            g = plots.get_subplot_plotter(subplot_size=7)
            g.settings.alpha_filled_add=0.6
            g.settings.solid_contour_palefactor = 0.7
            g.settings.alpha_factor_contour_lines = 1
            if names[dummy][i+1] in wanted_parameters :
                print("contour for om-",names[dummy][i+1])
                g.plot_2d(list_mc,names[dummy][0],names[dummy][i+1],filled=True,clear=False)
                xdata = mc[dummy].samples[:, mc[dummy].index[names[dummy][0]]]
                ydata = mc[dummy].samples[:, mc[dummy].index[names[dummy][i+1]]]
                # plt.xlim(xdata.min(),xdata.max())
                # plt.ylim(ydata.min(),ydata.max())
                g.add_legend(wanted_chains, legend_loc='upper right')
                plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(nbins=8))  # increase nbins
                plt.gca().yaxis.set_major_locator(ticker.MaxNLocator(nbins=8))
                # plt.savefig(f"{i}.pdf")

files = {"kids nl" : os.path.join(output_folder,"output_multinest_true.txt"),
         "kids lin" : os.path.join(output_folder,"output_multinest_kids_lin.txt"),
         "SN" : os.path.join(output_folder,"output_multinest_SN.txt"), 
         "SN+BAO" : os.path.join(output_folder,"output_multinest_SN_BAO.txt"), 
         "SN+BAO+QSO" : os.path.join(output_folder,"output_multinest_SN_BAO_QSO.txt"),
         "SN+BAO+QSO_z2" : os.path.join(output_folder,"output_multinest_SN_BAO_QSO_z2.txt"),
         "Marie" : os.path.join(output_folder,"output_multinest_SN_BAO_QSO_Marie.txt"),
         }
process = postprocess(files)
# process.best_fit()
wanted_params = ["\\Omega_m","\\sigma_8","S_8","\\Sigma_8"]
wanted_chains = ["SN","SN+BAO","SN+BAO+QSO","Marie"]
process.plot_contour(wanted_params,wanted_chains)


In [22]:
from scipy.integrate import trapezoid
c = 299792.458
h0 = 0.735
omh2 = 0.1424
om = omh2/h0**2

ze = 0.5
ae = 1/(1+ze)
a = np.linspace(ae,1.0,200)
model = model_independent(np.array([1]),a,h0,om,-1)

Da = ae*c/h0/100 * trapezoid(1/a**2/model.lcdm_expf(),a)
kh = 0.1
k = kh*h0
ell = Da * k
theta = 2*np.pi/ell*180/np.pi*60
print(theta, ell)

241.9428789305302 89.27727112895137


In [ ]:
## Pure cheb as expansion function
def model (a,*c) :
    h0 = 0.735 # 0.6766
    cheb_poly = np.polynomial.chebyshev.Chebyshev(c, domain=[0.0,1.0])
    da = (1-a[0])/a[0]
    x = (a-a[0])/(a[-1]-a[0])
    expf = da*h0/(1+da*x)**2/(cheb_poly(x))
    return expf/expf[-1]

In [ ]:
## normalized cheb, can be used as shifted cheb by : cheb(c,2x+1)
def cheb(c,a):
    x = np.arccos(a)
    result = 0
    for i in range(len(c)) :
        if i ==0 :
            result += (1/np.pi)**(1/2)*c[i]
        else :
            result += (2/np.pi)**(1/2)*c[i]*np.cos(i*x)
    return result

In [ ]:
## Alex' codes

import numpy as np 
import matplotlib.pyplot as plt 

x = np.linspace(0,1,100)
c = np.array([0.944,-0.357,0.047,0.0052])

a_min = 0.294118
h = 0.735
Om = 0.36

delta_a = (1-a_min)/a_min

def a(x):   
    return (1+delta_a*x)*a_min

def E(x,e): 
    E = delta_a/(1+delta_a*x)**2/e/h
    return  E / E[-1]

#this is what is implemented in the lastro, see also def. in https://scipost.org/SciPostAstro.2.1.001/pdf appendix A
def chebyshev(x, c):
    n = np.arccos(x)
    result = 0
    for i in range(len(c)): 
        if i == 0: 
            result += (1/np.pi)**(1/2)*c[i]
        else: 
            result += (2/np.pi)**(1/2)*c[i]*np.cos(i*n)
    return result

def E_lcdm(a):
    return np.sqrt(Om*a**(-3)+1-Om)

#numpy
cheb = np.polynomial.chebyshev.Chebyshev(c)

plt.figure()
plt.loglog(x, chebyshev(2*x-1,c), label='own impl.')
plt.loglog(x, cheb(2*x-1), label='numpy')
plt.legend()
plt.grid()

plt.figure()
plt.loglog(a(x), E(x,chebyshev(2*x-1,c)), label='own impl.')
# plt.loglog(a(x), E(x,cheb(2*x-1)),label='numpy')
plt.loglog(a(x), E_lcdm(a(x)),label='lcdm')
plt.legend()
plt.grid()
plt.show()


In [ ]:
# n=0 model

def model_expf (self, a=None, c=None) :
        if c is None :
            c = self.c
        if a is not None : # Needed for odeint
            x = (a-self.amin)/(self.amax-self.amin)
        else :
            x = self.x
        expf = self.da/(1+self.da*x)**2/self.chebs(c,2*x-1)/self.h0
        expf = expf/self.norm
        return expf
    
    def d_expansion_function_d_a (self,a=None,c=None):
        if c is None :
            c = self.c
        if a is not None : # Needed for odeint
            x = (a-self.amin)/(self.amax-self.amin)
        else :
            x = self.x
        A = (1+x*self.da)
        prefactor = -self.da/self.h0/(self.a[-1]-self.a[0])/A**2/self.chebs(c,2*x-1)
        term1 = 2*self.da/A
        term2 = 2*self.cheb_derivative(c,2*x-1)/self.chebs(c,2*x-1)
        d_expf = prefactor*(term1 + term2)/self.norm
        return d_expf

In [ ]:
# Testing if the new model is implemented correctly

c = np.array([0.119,-0.0349,0.0032])
c = np.array([-0.110, 0.02561])
a_new = np.linspace(0.001,1,1000)
new_model = model_independent(c,a_new,om=0.32,h0=0.735, n=-1)
new_model2 = model_independent(c,a_new,om=0.37,h0=0.735, n=-1)

plt.loglog(a_new, new_model.model_expf(),label="model")
plt.loglog(a_new, new_model.lcdm_expf(),label="LCDM om=0.32")
plt.loglog(a_new, new_model2.lcdm_expf(),label="LCDM om=0.37")

plt.legend()
plt.show()

test1 = model_independent(c,a,n=-1)
test2 = model_independent(c,a)
plt.plot(a,test1.d_expansion_function_d_a(),label="n=0")
plt.plot(a,test2.d_expansion_function_d_a(),label="n=-1")
plt.legend()

plt.plot(test1.a,test1.model_expf(),label="n=-1")
plt.plot(test1.a,test1.model_expf(n=0),label="n=0")
plt.plot(test1.a,test1.lcdm_expf(),label="lcdm")
plt.legend()
plt.show()

## Testing the implementation of Growth Factor
c = np.array([0.72196921, -0.15288923 ,-0.02065917 , 0.00706415])
zm = np.loadtxt("z_matterme.txt")
am = 1/(1+zm)[::-1]
dplus = Growth_Factor(c,am, n=-1)
dpl_kids = np.loadtxt('dpl_kids.txt')[::-1]
dpl_kids = dpl_kids/dpl_kids[0]
Dplus = []
Dpluss = []

for i in dplus.a :
    Dplus.append(dplus.growth_factor(i))
    # Dpluss.append(dplus.D_plus(i))
# Dpluss = np.array(Dpluss)
# Dpluss = Dpluss/Dpluss[0]
Dplus = np.array(Dplus)
Dplus = Dplus/Dplus[0]
plt.plot(dplus.a,Dplus,label='model',marker='.')
plt.plot(dplus.a,dpl_kids,label='lcdm',marker='.')
# plt.plot(dplus.a,Dpluss,label='model')
plt.legend()
# plt.plot(a,)
plt.show()

c = np.array([1.22204523e+00, -6.32980436e-01,  1.58064939e-01, -4.90222909e-02,
  2.31382692e-02, -1.04376390e-02,  4.24459960e-03, -1.52263287e-03,
  1.13066612e-03])
zm = np.loadtxt("z_matterme.txt")
am = 1/(1+zm)[::-1]
dplus = Growth_Factor(c,am, n=0)
dpl_kids = np.loadtxt('dpl_kids.txt')[::-1]
dpl_kids = dpl_kids/dpl_kids[0]
Dplus = []
Dpluss = []

for i in dplus.a :
    Dplus.append(dplus.growth_factor(i))
    # Dpluss.append(dplus.D_plus(i))
# Dpluss = np.array(Dpluss)
# Dpluss = Dpluss/Dpluss[0]
Dplus = np.array(Dplus)
Dplus = Dplus/Dplus[0]
plt.plot(dplus.a,Dplus,label='model',marker='.')
plt.plot(dplus.a,dpl_kids,label='lcdm',marker='.')
# plt.plot(dplus.a,Dpluss,label='model')
plt.legend()
# plt.plot(a,)
plt.show()

In [ ]:
# Look at the c_ell from all bin pairs

import numpy as np
ells = np.loadtxt("ellsme.txt")
cell = {}
for i in range(5):
    for j in range(i+1) :
        cell[f"{i+1}{j+1}"] = np.loadtxt(f"c_ell_lcdm_{i+1}_{j+1}.txt")
        plt.loglog(ells,cell[f"{i+1}{j+1}"],label=f"bin{i+1}{j+1}")
plt.legend()